# M6c · Build `mart_device_monthly_telemetry`

**Goal:** roll up `fact_harsh_event` and `fact_telemetry_daily` into a
single per-(tenant, device, year_month) mart that feeds the unified ML
view (`v_ml_features_full`).

**SQL file:** `sql/25_mart_device_monthly_telemetry.sql`.
Single parameter: `:touched_months` — pass `None` to recompute every month
present in the underlying facts (full backfill).

**Why a separate mart from `mart_device_monthly_behavior`?**
Keeping the original (tested) mart untouched protects existing baseline
risk-score consumers. The new mart is opt-in and joined at view time.

**Exit criteria:**
- One row per (tenant, device, year_month) for every device with archive
  pings in that month.
- `harsh_events_per_100km` reasonable when joined to `total_distance_km`
  (typical: 0–10 events / 100 km; outliers > 30 indicate threshold mis-calibration).


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs — preview the SQL and supporting fact counts

In [2]:
from accent_fleet.db.sql_loader import load_sql
print(load_sql('25_mart_device_monthly_telemetry.sql')[:900], '...')

import pandas as pd
with get_engine().connect() as conn:
    print('--- supporting fact counts ---')
    for t in ('fact_harsh_event', 'fact_telemetry_daily'):
        n = conn.execute(text(f'SELECT COUNT(*) FROM warehouse.{t}')).scalar_one()
        print(f'  warehouse.{t:25s}  {n:>10,}  rows')


-- =============================================================================
-- 25_mart_device_monthly_telemetry.sql
-- =============================================================================
-- mart_device_monthly_telemetry: companion to 20_mart_device_monthly_behavior.
-- One row per (tenant, device, year_month) carrying the archive-derived
-- features needed for Project 1 (Driver Behavior Scoring & Risk
-- Classification) — harsh-event counts, idling ratios, RPM signals, and
-- normalized "per-100km" rates.
--
-- Kept SEPARATE from the existing mart (rather than altering it) so the
-- existing tested SQL remains untouched. The unified ML view (sql/26)
-- LEFT JOINs the two on the natural key.
--
-- Parameter:
--   :touched_months  TEXT[]   e.g. ARRAY['2026-03','2026-04']
-- =============================================================================

CREATE TABLE IF NOT EXI ...
--- supporting fact counts ---
  warehouse.fact_harsh_event            1,885,828  rows
  wareho

## 3. Execute — full recompute (pass NULL touched_months)

In [3]:
from accent_fleet.db import run_sql_file, transaction
from accent_fleet.pipeline.run_log import begin_run, end_run
import time

run_id = begin_run(mode='notebook:03_build_mart_device_monthly_telemetry')
t0 = time.time()
try:
    with transaction() as conn:
        result = run_sql_file(
            conn,
            '25_mart_device_monthly_telemetry.sql',
            params={'touched_months': None, 'etl_run_id': run_id},
        )
        rows = result.rowcount or 0
    end_run(run_id, status='success', rows_loaded=rows)
    print(f'done in {time.time()-t0:.1f}s — rows: {rows:,}')
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise


done in 6.5s — rows: 2,593


## 4. Inspect — coverage and sample rows

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    coverage = pd.read_sql(text('''
        SELECT year_month,
               COUNT(*) AS rows,
               SUM(harsh_event_total) AS harsh_total,
               SUM(total_idle_minutes)::numeric(20,1) AS idle_minutes,
               AVG(monthly_idle_ratio)::numeric(6,3)  AS avg_idle_ratio
          FROM marts.mart_device_monthly_telemetry
         GROUP BY year_month
         ORDER BY year_month DESC LIMIT 12
    '''), conn)
    sample = pd.read_sql(text('SELECT * FROM marts.mart_device_monthly_telemetry ORDER BY harsh_event_total DESC LIMIT 5'), conn)
display(coverage); display(sample)
print('OK — mart_device_monthly_telemetry built.')


,year_month,rows,harsh_total,idle_minutes,avg_idle_ratio
0,2026-04,358,117682,111714.0,0.114
1,2026-03,367,338352,284373.0,0.110
2,2026-02,369,340995,276501.5,0.115
3,2026-01,381,306566,284989.5,0.130
4,2025-12,383,306346,316217.5,0.130
5,2025-11,390,292979,302658.5,0.113
6,2025-10,222,174062,237409.0,0.125
7,2025-09,5,1,6388.0,0.554
8,2025-08,3,0,7710.0,1.000
9,2025-07,3,0,1.0,1.000


,tenant_id,device_id,year_month,harsh_brake_count,harsh_accel_count,harsh_corner_count,harsh_event_total,harsh_moderate_count,harsh_high_count,harsh_extreme_count,...,active_telemetry_days,avg_rpm,max_rpm,total_high_rpm_seconds,high_rpm_minutes_per_day,avg_telemetry_speed_kmh,max_telemetry_speed_kmh,total_fuel_used_archive,_etl_run_id,_computed_at
0,7486,820420,2026-02,2313,456,34631,37400,15971,17142,4287,...,28,0.0,0,0.0,0.0,23.936336,105,0.0,35,2026-04-29 23:08:56.745649+00:00
1,7486,820420,2026-01,1759,363,32470,34592,13300,17554,3738,...,31,0.0,0,0.0,0.0,20.477280,104,0.0,35,2026-04-29 23:08:56.745649+00:00
2,7486,820420,2026-03,1870,408,27535,29813,12192,13975,3646,...,31,0.0,0,0.0,0.0,24.770267,103,0.0,35,2026-04-29 23:08:56.745649+00:00
3,7486,820420,2025-12,1504,338,27209,29051,11956,13842,3253,...,31,0.0,0,0.0,0.0,21.418855,95,0.0,35,2026-04-29 23:08:56.745649+00:00
4,235,425193,2025-10,6996,6556,12739,26291,10037,7986,8268,...,31,469.0,469,0.0,0.0,23.007863,129,0.0,35,2026-04-29 23:08:56.745649+00:00


OK — mart_device_monthly_telemetry built.
